# 1. Initial Margin and Baseline Counterparty Risk

This notebook derives one-sided 99% delta-normal IM and simulates the 252-day observation period. An additional MPOR segment is generated only to observe close-out after late-window defaults.

## Initial margin derivation

For $V_t=Q(S_t-K)$ and arithmetic Brownian motion, $\Delta S\sim N(0,\sigma_s^2\Delta t)$. Thus $IM=z_{0.99}Q\sigma_s\sqrt{MPOR/252}$, with dollar volatility $\sigma_s=20\%\times S_0$.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.otc_margining import ModelConfig,simulate_prices,simulate_default_days,simulate_liquidity,summarize_simulation
cfg=ModelConfig(paths=20000,horizon=252,maturity=300,mpor=10,seed=42)
prices=simulate_prices(cfg,seed=100,extra_days=cfg.mpor)
default_days=simulate_default_days(cfg,seed=200)
print(f'99% initial margin: ${cfg.initial_margin():,.2f}')
print(f'Daily exogenous PD: {cfg.daily_pd:.8%}; cumulative over 252 days: {1-(1-cfg.daily_pd)**cfg.horizon:.2%}')

In [ ]:
fig,ax=plt.subplots(figsize=(9,4.5))
for i in range(50): ax.plot(prices[i,:cfg.horizon+1],alpha=.25,lw=.7)
ax.axhline(cfg.strike,color='black',ls='--',label='Forward strike')
ax.set(xlabel='Trading day',ylabel='Underlying price ($)',title='Sample arithmetic-Brownian price paths')
ax.legend(frameon=False);plt.show()
base=simulate_liquidity(prices,default_days,cfg,mode='unlimited')
pd.Series(summarize_simulation(base,cfg))